# Model Training and Experiment Tracking

This notebook trains multiple credit risk classification models, performs hyperparameter tuning, tracks experiments with MLflow, and evaluates model performance.

**Objectives:**
1. Load and prepare training/test data
2. Train multiple classification models
3. Perform hyperparameter tuning
4. Track experiments with MLflow
5. Evaluate and compare models
6. Register best model in MLflow Model Registry

## 1. Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import mlflow
import mlflow.sklearn

print(f"MLflow Version: {mlflow.__version__}")

## 2. Load and Prepare Data

In [ ]:
# Load processed data
df = pd.read_csv('../data/processed/processed_features.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df['is_high_risk'].value_counts())
print(f"\nClass balance: {(df['is_high_risk'].sum() / len(df) * 100):.2f}% high-risk")

In [ ]:
# Prepare features and target
X = df.drop('is_high_risk', axis=1)
y = df['is_high_risk']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set target distribution:")
print(y_train.value_counts())
print(f"\nTest set target distribution:")
print(y_test.value_counts())

## 3. Define Model Evaluation Function

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evaluate model on train and test sets
    """
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Probabilities for ROC-AUC
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    metrics = {
        'train_accuracy': accuracy_score(y_train, y_train_pred),
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'train_precision': precision_score(y_train, y_train_pred, zero_division=0),
        'test_precision': precision_score(y_test, y_test_pred, zero_division=0),
        'train_recall': recall_score(y_train, y_train_pred, zero_division=0),
        'test_recall': recall_score(y_test, y_test_pred, zero_division=0),
        'train_f1': f1_score(y_train, y_train_pred, zero_division=0),
        'test_f1': f1_score(y_test, y_test_pred, zero_division=0),
        'train_roc_auc': roc_auc_score(y_train, y_train_proba),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba)
    }
    
    return metrics

print("Evaluation function defined.")

## 4. Train Logistic Regression with Hyperparameter Tuning

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("Credit_Risk_Classification")

# Logistic Regression with GridSearch
print("Training Logistic Regression...")

with mlflow.start_run(run_name="LogisticRegression_GridSearch"):
    lr_params = {
        'C': [0.001, 0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs'],
        'max_iter': [200]
    }
    
    lr = LogisticRegression(random_state=42)
    lr_grid = GridSearchCV(lr, lr_params, cv=5, scoring='roc_auc', n_jobs=-1)
    lr_grid.fit(X_train, y_train)
    
    best_lr = lr_grid.best_estimator_
    lr_metrics = evaluate_model(best_lr, X_train, X_test, y_train, y_test, 'LogisticRegression')
    
    # Log parameters and metrics
    mlflow.log_params(lr_grid.best_params_)
    mlflow.log_metrics(lr_metrics)
    mlflow.log_metric('best_cv_score', lr_grid.best_score_)
    
    # Log model
    mlflow.sklearn.log_model(best_lr, "logistic_regression_model")
    
    print(f"\nLogistic Regression Results:")
    print(f"Best Parameters: {lr_grid.best_params_}")
    print(f"Best CV Score: {lr_grid.best_score_:.4f}")
    print(f"Test ROC-AUC: {lr_metrics['test_roc_auc']:.4f}")
    print(f"Test F1 Score: {lr_metrics['test_f1']:.4f}")

## 5. Train Gradient Boosting with Hyperparameter Tuning

In [ ]:
# Gradient Boosting with RandomSearch
print("Training Gradient Boosting...")

with mlflow.start_run(run_name="GradientBoosting_RandomSearch"):
    gb_params = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [2, 4]
    }
    
    gb = GradientBoostingClassifier(random_state=42)
    gb_random = RandomizedSearchCV(
        gb, gb_params, n_iter=10, cv=5, scoring='roc_auc', 
        random_state=42, n_jobs=-1
    )
    gb_random.fit(X_train, y_train)
    
    best_gb = gb_random.best_estimator_
    gb_metrics = evaluate_model(best_gb, X_train, X_test, y_train, y_test, 'GradientBoosting')
    
    # Log parameters and metrics
    mlflow.log_params(gb_random.best_params_)
    mlflow.log_metrics(gb_metrics)
    mlflow.log_metric('best_cv_score', gb_random.best_score_)
    
    # Log model
    mlflow.sklearn.log_model(best_gb, "gradient_boosting_model")
    
    print(f"\nGradient Boosting Results:")
    print(f"Best Parameters: {gb_random.best_params_}")
    print(f"Best CV Score: {gb_random.best_score_:.4f}")
    print(f"Test ROC-AUC: {gb_metrics['test_roc_auc']:.4f}")
    print(f"Test F1 Score: {gb_metrics['test_f1']:.4f}")

## 6. Train Random Forest Model

In [ ]:
# Random Forest
print("Training Random Forest...")

with mlflow.start_run(run_name="RandomForest"):
    rf_params = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [2, 4]
    }
    
    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    rf_random = RandomizedSearchCV(
        rf, rf_params, n_iter=10, cv=5, scoring='roc_auc', 
        random_state=42, n_jobs=-1
    )
    rf_random.fit(X_train, y_train)
    
    best_rf = rf_random.best_estimator_
    rf_metrics = evaluate_model(best_rf, X_train, X_test, y_train, y_test, 'RandomForest')
    
    # Log parameters and metrics
    mlflow.log_params(rf_random.best_params_)
    mlflow.log_metrics(rf_metrics)
    mlflow.log_metric('best_cv_score', rf_random.best_score_)
    
    # Log model
    mlflow.sklearn.log_model(best_rf, "random_forest_model")
    
    print(f"\nRandom Forest Results:")
    print(f"Best Parameters: {rf_random.best_params_}")
    print(f"Best CV Score: {rf_random.best_score_:.4f}")
    print(f"Test ROC-AUC: {rf_metrics['test_roc_auc']:.4f}")
    print(f"Test F1 Score: {rf_metrics['test_f1']:.4f}")

## 7. Model Comparison and Selection

In [ ]:
# Compare models
models_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Gradient Boosting', 'Random Forest'],
    'Test Accuracy': [lr_metrics['test_accuracy'], gb_metrics['test_accuracy'], rf_metrics['test_accuracy']],
    'Test Precision': [lr_metrics['test_precision'], gb_metrics['test_precision'], rf_metrics['test_precision']],
    'Test Recall': [lr_metrics['test_recall'], gb_metrics['test_recall'], rf_metrics['test_recall']],
    'Test F1 Score': [lr_metrics['test_f1'], gb_metrics['test_f1'], rf_metrics['test_f1']],
    'Test ROC-AUC': [lr_metrics['test_roc_auc'], gb_metrics['test_roc_auc'], rf_metrics['test_roc_auc']]
})

print("\nModel Comparison:")
print(models_comparison.to_string(index=False))

# Find best model by ROC-AUC
best_model_idx = models_comparison['Test ROC-AUC'].idxmax()
best_model_name = models_comparison.loc[best_model_idx, 'Model']
best_roc_auc = models_comparison.loc[best_model_idx, 'Test ROC-AUC']

print(f"\nBest Model: {best_model_name} (ROC-AUC: {best_roc_auc:.4f})")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

metrics_to_plot = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1 Score', 'Test ROC-AUC']
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    axes[idx].bar(models_comparison['Model'], models_comparison[metric])
    axes[idx].set_title(metric)
    axes[idx].set_ylabel('Score')
    axes[idx].set_ylim([0, 1])
    axes[idx].tick_params(axis='x', rotation=45)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 8. Best Model Detailed Evaluation

In [ ]:
# Select best model
if best_model_name == 'Logistic Regression':
    best_model = best_lr
    best_metrics = lr_metrics
elif best_model_name == 'Gradient Boosting':
    best_model = best_gb
    best_metrics = gb_metrics
else:
    best_model = best_rf
    best_metrics = rf_metrics

print(f"Best Model: {best_model_name}")
print(f"\nDetailed Metrics:")
for metric, value in best_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Confusion matrix and classification report
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'])
ax.set_title(f'{best_model_name} - Confusion Matrix')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'{best_model_name} - ROC Curve')
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Save Best Model and Register with MLflow

In [ ]:
# Save best model
model_path = '../models/best_model.pkl'
joblib.dump(best_model, model_path)
print(f"Best model saved to: {model_path}")

# Register model in MLflow
model_uri = f"runs:/{mlflow.active_run().info.run_id}/best_model"
model_details = mlflow.register_model(model_uri, "CreditRiskModel")
print(f"\nModel registered with MLflow:")
print(f"  Name: CreditRiskModel")
print(f"  Version: {model_details.version}")
print(f"  Stage: {model_details.current_stage}")

In [ ]:
# Transition model to production
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name="CreditRiskModel",
    version=model_details.version,
    stage="Production"
)
print(f"\nModel transitioned to Production stage")

## 10. Summary and Next Steps

✅ **Completed:**
- Trained 3 classification models: Logistic Regression, Gradient Boosting, Random Forest
- Performed hyperparameter tuning using GridSearch and RandomSearch
- Tracked all experiments with MLflow
- Evaluated models using multiple metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
- Selected and registered best model: **{best_model_name}**

**Next Steps:**
1. Deploy model to production API (src/api/main.py)
2. Create unit tests (tests/test_data_processing.py)
3. Set up CI/CD pipeline (.github/workflows/ci.yml)
4. Containerize with Docker for deployment
5. Monitor model performance in production